In [1]:
import pandas as pd
import geopandas as gpd
from functions import *

Load data (https://www.istat.it/notizia/confini-delle-unita-amministrative-a-fini-statistici-al-1-gennaio-2018-2/)

In [2]:
gdf = gpd.read_file('data_origin/geo_data/mun_perimeters/Com01012025_WGS84.shp')

Rename + select columns

In [3]:
# rename PRO_COM_T in mun_istat
gdf = gdf.rename(columns = {
    'PRO_COM_T' : 'mun_istat',
    'COMUNE' : 'mun_name'
})

col_to_keep = [
    'mun_name',
    'geometry',
    'mun_istat'
]

gdf = gdf[col_to_keep]

Upload ISTAT CODES

In [14]:
df_new_istat = pd.read_csv('datasets/mun_istat_codes.csv')

df_new_istat = df_new_istat[['mun_istat','mun_name']]

df_change = pd.read_csv('datasets/changes_istat.csv')

# Uniform ISTAT codes across datasets
add_zeroes(df_new_istat, ['mun_istat'], 6)
add_zeroes(df_change, ['mun_istat_old', 'mun_istat_new'], 6)

df_new_istat['mun_istat'] = df_new_istat['mun_istat'].astype('object')
df_change['mun_istat_old'] = df_change['mun_istat_old'].astype('object')
df_change['mun_istat_new'] = df_change['mun_istat_new'].astype('object')

In [21]:
# Update mun istat
gdf_updated = update_istat(
    df=gdf,
    df_map=df_change, 
    valid_codes=df_new_istat["mun_istat"], 
    istat_col="mun_istat",
    istat_old = "mun_istat_old",
    istat_new = "mun_istat_new"
)

gdf_updated['mun_name'] = gdf_updated['mun_name'].apply(normalize_name)

gdf_updated = gdf_updated.drop(columns = ['mun_istat'])

In [22]:
# Check municipality with no match
suppressed_df = gdf_updated[gdf_updated['suppressed'] == True].copy()
non_suppressed_df = gdf_updated[gdf_updated['suppressed'] == False].copy()

similarity = similarity_score(suppressed_df, df_new_istat, col = 'mun_name')
similarity

,Name in df1,Name in df2,Similarity score (0-100)
1,NUGHEDU SAN NICOLA2,NUGHEDU SAN NICOLO,91.891892
2,TORTOLA,TORTONA,85.714286
0,BUDDUSA2,BUDDUSO,80.000000


In [23]:
suppressed_df['mun_name'] = suppressed_df['mun_name'].replace({
	'NUGHEDU SAN NICOLA2' : 'NUGHEDU SAN NICOLO',
	'TORTOLA' : 'TORTOLI',
	'BUDDUSA2' : 'BUDDUSO'
})

suppressed_df = pd.merge(suppressed_df, df_new_istat, on = ['mun_name'], how = 'left')

# Replace ISTAT code in suppressed_df
suppressed_df['mun_istat_updated'] = suppressed_df['mun_istat']

# Drop helper column
suppressed_df = suppressed_df.drop(columns=['mun_istat'])

# Concatenate with non-suppressed rows
gdf_updated = pd.concat([non_suppressed_df, suppressed_df], ignore_index=True)

# Adjust columns
gdf_updated = gdf_updated.drop(columns = ['changed','suppressed'])

gdf_updated = gdf_updated.rename(columns = {'mun_istat_updated' : 'mun_istat'})

In [25]:
gdf_updated.loc[gdf_updated['mun_istat'] == '104009', 'mun_istat'] = '114043'

Save Municipality areas

In [27]:
# Import Province and Region names
df_name = pd.read_csv('datasets/names_corr.csv')
add_zeroes(df_name, 'mun_istat', 6)

# Import names into geodataframe
gdf_updated = pd.merge(gdf_updated, df_name.drop(columns = 'mun_name'), on = 'mun_istat', how = 'left')

# Fill nan values
mapping = {
    '016215': ('BERGAMO', 'LOMBARDIA'),
    '051041': ('AREZZO', 'TOSCANA'),
    '061102' : ('CASERTA', 'CAMPANIA'),
    '071064' : ('FOGGIA', 'PUGLIA'),
    '078132' : ('COSENZA', 'CALABRIA'),
    '079073' : ('CATANZARO', 'CALABRIA'),
    '081010' : ('TRAPANI', 'SICILIA'),
    '113009' : ('GALLURA NORDEST SARDEGNA', 'SARDEGNA'),
    '113021' : ('GALLURA NORDEST SARDEGNA', 'SARDEGNA'),
    '110005' : ('BARLETTA - ANDRIA - TRANI', 'PUGLIA'),
    '113026' : ('SASSARI', 'SARDEGNA'),
    '113013' : ('SASSARI', 'SARDEGNA'),
    '113023' : ('GALLURA NORDEST SARDEGNA', 'SARDEGNA'),
    '116005' : ('OGLIASTRA', 'SARDEGNA'),
    '115044' : ('ORISTANO', 'SARDEGNA'),
    '115029' : ('ORISTANO', 'SARDEGNA')
}

for k, (prov, reg) in mapping.items():
    mask = gdf_updated['mun_istat'] == k
    gdf_updated.loc[mask, ['prov_name', 'reg_name']] = [prov, reg]

In [28]:
# Import non-normalised names
corr_df = pd.read_csv('datasets/names_corr_non_normalised.csv')
add_zeroes(corr_df, 'mun_istat', 6)

# Drop normalised names
gdf_updated = gdf_updated.drop(columns = ['mun_name', 'prov_name', 'reg_name'])

# Merge
gdf_updated = gpd.GeoDataFrame(pd.merge(gdf_updated, corr_df, on = 'mun_istat', how = 'left'))

# Import info for missing municipality (None, Torino, Piemonte)
gdf_updated.loc[(gdf_updated['mun_istat'] == '001168'), ['mun_name', 'prov_name', 'reg_name']] = ['None', 'Torino', 'Piemonte']

# Drop remaining null values
gdf_updated = gdf_updated.dropna()

In [ ]:
gdf_updated.to_file('datasets/geo_data/mun_perimeters.gpkg', layer = 'mun_perimeters', driver = 'GPKG')

Dissolve Province/Region

In [31]:
# Dissolve Province areas
gdf_prov = gdf_updated.dissolve(by = 'prov_name').reset_index()
gdf_prov = gdf_prov.drop(columns = ['mun_name', 'mun_istat'])

# Dissolve Region areas
gdf_reg = gdf_prov.dissolve(by = 'reg_name').reset_index()
gdf_reg = gdf_reg.drop(columns = 'prov_name')

In [ ]:
gdf_prov.to_file('datasets/geo_data/prov_perimeters.gpkg', layer= 'prov_perimeters', driver = 'GPKG')

gdf_reg.to_file('datasets/geo_data/reg_perimeters.gpkg', layer= 'reg_perimeters', driver = 'GPKG')